# Analyse de la durée et de la consommation en phase de taxi au sol

Ce notebook analyse la durée et la consommation de carburant lors de la phase de taxi au sol à partir du fichier **taxi_gold_enriched.csv**.

## 1. Importer les bibliothèques nécessaires
Nous allons utiliser pandas, numpy, matplotlib et seaborn pour l'analyse et la visualisation des données.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage des graphiques dans le notebook
%matplotlib inline

## 2. Charger et explorer le fichier taxi gold
Nous allons lire le fichier `taxi_gold_enriched.csv` et afficher les premières lignes pour comprendre la structure des données.

## 3. Calculer la durée de taxi au sol
La durée de taxi au sol est donnée par la colonne `Duree_min` (en minutes).

In [ ]:
# Statistiques descriptives sur la durée de taxi au sol
df['Duree_min'].describe()

## 4. Analyser la consommation de carburant pendant le taxi
La consommation de carburant pendant le taxi est donnée par la colonne `Consommation_totale_kg` (en kilogrammes).

In [ ]:
# Statistiques descriptives sur la consommation de carburant pendant le taxi
df['Consommation_totale_kg'].describe()

## 5. Visualiser la relation entre durée et consommation
Nous allons créer des graphiques pour visualiser la relation entre la durée de taxi (`Duree_min`) et la consommation de carburant (`Consommation_totale_kg`).

In [ ]:
# Nuage de points durée vs consommation
plt.figure(figsize=(8,6))
sns.scatterplot(x='Duree_min', y='Consommation_totale_kg', data=df)
plt.xlabel('Durée taxi au sol (min)')
plt.ylabel('Consommation carburant (kg)')
plt.title('Relation entre durée de taxi et consommation de carburant')
plt.show()

# Histogramme des durées de taxi
plt.figure(figsize=(8,4))
sns.histplot(df['Duree_min'], bins=30, kde=True)
plt.xlabel('Durée taxi au sol (min)')
plt.title('Distribution des durées de taxi au sol')
plt.show()

# Histogramme de la consommation
plt.figure(figsize=(8,4))
sns.histplot(df['Consommation_totale_kg'], bins=30, kde=True)
plt.xlabel('Consommation carburant (kg)')
plt.title('Distribution de la consommation de carburant pendant le taxi')
plt.show()

In [ ]:
# Charger le fichier taxi_gold_enriched.csv
file_path = 'notebooks/data/taxi_gold_enriched.csv'
df = pd.read_csv(file_path)
df.head(), df.columns

## 3. Calculer la durée de taxi au sol
Nous allons extraire les colonnes pertinentes (heures de début et de fin du taxi) et calculer la durée de chaque phase de taxi au sol.

In [ ]:
# Supposons que les colonnes s'appellent 'taxi_out_start', 'taxi_out_end' (à adapter selon le fichier)
# Conversion en datetime

df['taxi_out_start'] = pd.to_datetime(df['taxi_out_start'])
df['taxi_out_end'] = pd.to_datetime(df['taxi_out_end'])

df['taxi_out_duration_min'] = (df['taxi_out_end'] - df['taxi_out_start']).dt.total_seconds() / 60

df[['taxi_out_start', 'taxi_out_end', 'taxi_out_duration_min']].head()

## 4. Analyser la consommation de carburant pendant le taxi
Nous allons calculer la consommation de carburant pour chaque phase de taxi en utilisant les données disponibles (par exemple, une colonne 'fuel_used_taxi').

In [ ]:
# Supposons que la colonne de consommation s'appelle 'fuel_used_taxi' (à adapter selon le fichier)

df['fuel_used_taxi'].describe()

## 5. Visualiser la relation entre durée et consommation
Nous allons créer des graphiques pour visualiser la relation entre la durée de taxi et la consommation de carburant.

In [ ]:
# Nuage de points durée vs consommation
plt.figure(figsize=(8,6))
sns.scatterplot(x='taxi_out_duration_min', y='fuel_used_taxi', data=df)
plt.xlabel('Durée taxi au sol (min)')
plt.ylabel('Consommation carburant (kg)')
plt.title('Relation entre durée de taxi et consommation de carburant')
plt.show()

# Histogramme des durées de taxi
plt.figure(figsize=(8,4))
sns.histplot(df['taxi_out_duration_min'], bins=30, kde=True)
plt.xlabel('Durée taxi au sol (min)')
plt.title('Distribution des durées de taxi au sol')
plt.show()

# Histogramme de la consommation
plt.figure(figsize=(8,4))
sns.histplot(df['fuel_used_taxi'], bins=30, kde=True)
plt.xlabel('Consommation carburant (kg)')
plt.title('Distribution de la consommation de carburant pendant le taxi')
plt.show()

# J1 – Analyse descriptive

Dans cette partie, nous allons explorer en détail les variables du fichier `taxi_gold_enriched.csv`, vérifier la qualité des données, produire des statistiques descriptives et visualiser les relations importantes.

## 1. Lecture et aperçu des données
On relit le fichier pour garantir la fraîcheur des données et on affiche les premières lignes et les types de variables.

In [ ]:
# Lecture du fichier
df = pd.read_csv('notebooks/data/taxi_gold_enriched.csv')
df.head(), df.info()

## 2. Description des variables importantes
Nous allons décrire les variables clés :
- `Duree_min` : durée de taxi au sol (minutes)
- `Consommation_totale_kg` : consommation totale de carburant (kg)
- `Q_1 [lb/h]`, `Q_2 [lb/h]` : débits moteurs 1 et 2
- `AntiIce_ON` : système anti-givre (0 = off, 1 = on)
- Autres variables pertinentes selon le contexte

In [ ]:
# Aperçu statistique des variables importantes
variables = ['Duree_min', 'Consommation_totale_kg', 'Q_1 [lb/h]', 'Q_2 [lb/h]', 'AntiIce_ON']
df[variables].describe().T

In [ ]:
# Vérification des valeurs manquantes
missing = df[variables].isnull().sum()
missing

In [ ]:
# Détection des outliers par l'écart interquartile (IQR)
def detect_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((series < lower) | (series > upper)).sum()

outliers = {var: detect_outliers(df[var]) for var in variables if df[var].dtype != 'O'}
outliers

In [ ]:
# Statistiques avancées : médiane, variance, corrélations
stats = df[variables].agg(['mean', 'median', 'var'])
corr = df[variables].corr()
stats, corr

In [ ]:
# Visualisations descriptives
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df['Duree_min'], bins=30, kde=True, ax=axs[0,0])
axs[0,0].set_title('Histogramme Durée taxi au sol')
sns.histplot(df['Consommation_totale_kg'], bins=30, kde=True, ax=axs[0,1])
axs[0,1].set_title('Histogramme Consommation totale')
sns.scatterplot(x='Duree_min', y='Consommation_totale_kg', data=df, ax=axs[1,0])
axs[1,0].set_title('Durée vs Consommation')
sns.boxplot(x='AntiIce_ON', y='Consommation_totale_kg', data=df, ax=axs[1,1])
axs[1,1].set_title('Consommation selon Anti-Ice')
plt.tight_layout()
plt.show()

# Heatmap des corrélations
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Matrice de corrélation')
plt.show()

## Commentaires sur l'analyse descriptive

- Commentez ici les tendances observées :
  - Relation entre durée et consommation
  - Effet du système anti-givre
  - Corrélations principales
  - Présence d’outliers ou de valeurs manquantes

---

# J2 – Modélisation et validation

Dans cette partie, nous allons formuler des hypothèses, construire un modèle de régression pour prédire la consommation, et évaluer la validité des hypothèses.

## 1. Hypothèses
- La consommation est proportionnelle à la durée de taxi.
- L’activation de l’anti-givre augmente la consommation.
- Le débit moteur est corrélé à la consommation.

Nous allons tester ces hypothèses par la modélisation.

In [ ]:
# Préparation des données pour la régression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df[['Duree_min', 'Q_1 [lb/h]', 'Q_2 [lb/h]', 'AntiIce_ON']]
y = df['Consommation_totale_kg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Évaluation du modèle
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)

r2, rmse, mae

In [ ]:
# Visualisation des prédictions vs valeurs réelles
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Consommation réelle (kg)')
plt.ylabel('Consommation prédite (kg)')
plt.title('Prédictions vs Réel')
plt.show()

## Conclusion sur la modélisation

- Interprétez ici les résultats du modèle (qualité de la prédiction, validité des hypothèses, facteurs les plus influents, limites éventuelles, etc.).
- Proposez des pistes d’amélioration ou d’analyse complémentaire si besoin.